# Train/Test Split & Cross-Validation (Beginner)

When you train a model, the number you really care about is: **how well will it do on data it has never seen?** You can't just check accuracy on the same rows you trained on — the model has already memorised those, so that score is optimistic.

The classic fix is to **hold out** part of the data as a *test set*, train on the rest, and score on the held-out part. But there's a catch this notebook is all about: **which rows land in the test set is a random choice, and that choice moves your score around.** One split gives you one noisy number.

We'll show that instability directly, then meet **k-fold cross-validation (CV)**, which averages over many splits to give a steadier, more trustworthy estimate.

**Roadmap**
1. **Part 1 — one split is noisy:** train the *same* model many times, changing only the split, and watch the test accuracy jump around.
2. **Part 2 — k-fold CV:** reuse every row for both training and validation, and report `mean ± std`.
3. **Compare:** overlay the two and talk about *bias* and *variance* of the estimate.

In [ ]:
import numpy as np                 # numeric arrays + basic stats (mean, std, min, max)
import pandas as pd                # tidy tables, handy for collecting results
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # prettier statistical plots (histogram / strip plot)

# scikit-learn pieces we need:
from sklearn.datasets import load_breast_cancer          # a small, built-in classification dataset
from sklearn.model_selection import (
    train_test_split,   # cut data into one train part and one test part
    cross_val_score,    # run k-fold CV and return one score per fold
    KFold,              # plain k-fold splitter
    StratifiedKFold,    # k-fold that keeps class balance in every fold (best for classification)
)
from sklearn.linear_model import LogisticRegression      # a simple, fast, well-behaved classifier

# One global seed for anything ad-hoc. NOTE: the results below are made reproducible by the
# explicit random_state=... arguments we pass in each call, so this line is just a safe default.
np.random.seed(0)

sns.set_theme(style="whitegrid")   # clean seaborn look for all plots
print("imports ok")

## The idea of a train/test split

Imagine studying for an exam. If the teacher tests you on the *exact* practice questions you already saw the answers to, a high score doesn't prove you learned the subject — it proves you memorised those questions. To measure real understanding, the teacher keeps some **fresh** questions hidden until the exam.

A machine-learning model is the same:

- **Training set** → the practice questions the model learns from.
- **Test set** → fresh, held-out rows the model never saw, used only to score it.

`train_test_split` does this cut for us. It shuffles the rows and slices off a chunk (say 25%) for testing. The shuffle is controlled by a **`random_state`** number: change that number and a *different* set of rows becomes the test set.

That last point is the whole story of Part 1. Because the test set is a small random sample, the accuracy you measure on it is itself a bit random. **Trusting a single split is like judging a student on a single 5-question quiz — unlucky question picks can swing the grade.**

In [ ]:
# Load the Breast Cancer Wisconsin dataset that ships inside scikit-learn (no download needed).
# It's a binary classification problem: predict whether a tumour is malignant or benign
# from 30 numeric measurements of the cell nuclei.
data = load_breast_cancer()
X = data.data       # shape (569, 30): the feature measurements, one row per patient sample
y = data.target     # shape (569,):    the label, 0 = malignant, 1 = benign

print(f"samples (rows): {X.shape[0]}")
print(f"features (cols): {X.shape[1]}")
print(f"classes: {data.target_names}  ->  label values present: {np.unique(y)}")

# How balanced are the two classes? (matters for stratification later)
# np.bincount counts how many times each label 0,1,... appears.
counts = np.bincount(y)
print(f"class balance -> malignant(0): {counts[0]}, benign(1): {counts[1]}")

## Part 1 — One split is a noisy estimate

Here is the experiment. We keep **everything** fixed — same dataset, same model, same test-set *size* (25%) — and change **only one thing**: the `random_state` that decides *which* rows get held out.

If a single split were a reliable measure of the model, every `random_state` would give roughly the same accuracy. Spoiler: it won't. We'll collect the test accuracy for 200 different splits and look at how far apart they land.

In [ ]:
n_repeats = 200                      # try 200 different random splits
split_accuracies = []                # we'll append one test accuracy per split here

for seed in range(n_repeats):        # seed = 0, 1, 2, ... , 199  (each gives a different split)
    # Cut the SAME data a fresh way. stratify=y keeps the malignant/benign ratio in both parts,
    # so the only thing changing between iterations is *which* rows land where.
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=0.25,              # hold out 25% for testing
        random_state=seed,           # <-- the ONLY knob we vary
        stratify=y,                  # preserve class balance in train and test
    )

    # Train a fresh model on this split's training rows.
    # max_iter is bumped up just so the optimiser fully converges (avoids a warning); it does not
    # affect the point of the experiment.
    model = LogisticRegression(max_iter=5000)
    model.fit(X_tr, y_tr)

    # Score on THIS split's held-out test rows and remember the number.
    acc = model.score(X_te, y_te)    # .score() returns accuracy for a classifier
    split_accuracies.append(acc)

split_accuracies = np.array(split_accuracies)   # turn the list into an array for easy stats
print(f"collected {len(split_accuracies)} test accuracies (one per split)")

In [ ]:
# Summarise how much the score moved around just from re-splitting.
acc_min  = split_accuracies.min()
acc_max  = split_accuracies.max()
acc_mean = split_accuracies.mean()
acc_std  = split_accuracies.std()
spread   = acc_max - acc_min          # the full gap between the luckiest and unluckiest split

print("Single train/test split, same model, 200 different random_states")
print(f"  lowest  accuracy : {acc_min:.4f}")
print(f"  highest accuracy : {acc_max:.4f}")
print(f"  spread (max-min) : {spread:.4f}   <-- pure luck of the split!")
print(f"  mean +/- std     : {acc_mean:.4f} +/- {acc_std:.4f}")

# Read that spread out loud: depending only on which rows were held out, the SAME model looks
# several percentage points better or worse. If you had run the split once and reported that one
# number, you might have quoted the best case (or the worst) purely by chance.

In [ ]:
# Visualise the wobble two ways: a histogram (shape of the distribution) and a strip plot
# (every individual split as a dot). Both tell the same story from different angles.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram of the 200 accuracies.
sns.histplot(split_accuracies, bins=20, kde=True, color="#6699cc", ax=axes[0])
axes[0].axvline(acc_mean, color="black", linestyle="--", label=f"mean = {acc_mean:.3f}")
axes[0].set_title("Test accuracy over 200 random splits")
axes[0].set_xlabel("test accuracy")
axes[0].legend()

# Right: strip plot -- one jittered dot per split, so you can feel the range.
sns.stripplot(x=split_accuracies, color="#6699cc", alpha=0.5, jitter=0.3, ax=axes[1])
axes[1].axvline(acc_min, color="red",   linestyle=":", label=f"min = {acc_min:.3f}")
axes[1].axvline(acc_max, color="green", linestyle=":", label=f"max = {acc_max:.3f}")
axes[1].set_title("Each dot = one train/test split")
axes[1].set_xlabel("test accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

# Takeaway: the SAME model, SAME data, SAME split size -- yet the reported accuracy spans a visible
# range. A single split is one draw from this distribution, not the truth.

## Part 2 — k-fold cross-validation: a steadier estimate

Part 1's problem: a single split *wastes* data (the test rows never help training) and gives *one* noisy number. **k-fold cross-validation** fixes both by rotating the test set.

### The recipe (small diagram in words)
Split the data into **k** equal chunks called *folds* (say `k = 5`). Then run **k rounds**. Each round, one fold plays the role of the test set and the other `k-1` folds are the training set:

```
Round 1:  [TEST ][train][train][train][train]
Round 2:  [train][TEST ][train][train][train]
Round 3:  [train][train][TEST ][train][train]
Round 4:  [train][train][train][TEST ][train]
Round 5:  [train][train][train][train][TEST ]
```

Two things fall out of this:

1. **Every row is used for validation exactly once, and for training `k-1` times.** No data is wasted.
2. You get **k accuracy scores** instead of one. Their **mean** is your headline estimate; their **standard deviation** tells you how much it wobbles.

We report the result as $\text{mean} \pm \text{std}$:

$$\bar{a} = \frac{1}{k}\sum_{i=1}^{k} a_i \qquad s = \sqrt{\frac{1}{k}\sum_{i=1}^{k}(a_i - \bar{a})^2}$$

For **classification** we prefer **`StratifiedKFold`**, which makes each fold keep the same class ratio as the full dataset — otherwise a fold could accidentally get too few of one class and give a misleading score.

In [ ]:
# Build a 5-fold splitter. StratifiedKFold keeps the malignant/benign ratio steady in each fold.
# shuffle=True mixes the rows before splitting; random_state makes that shuffle reproducible.
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)

# The same kind of model as in Part 1 -- fresh, untrained. cross_val_score will clone and
# refit it once per fold internally, so we don't call .fit() ourselves.
cv_model = LogisticRegression(max_iter=5000)

# cross_val_score does the whole loop for us: for each of the k folds it trains on k-1 folds,
# scores on the held-out fold, and returns that fold's score. Result = array of k accuracies.
cv_scores = cross_val_score(cv_model, X, y, cv=skf, scoring="accuracy")

print(f"{k}-fold StratifiedKFold accuracies, one per fold:")
for i, s in enumerate(cv_scores, start=1):
    print(f"  fold {i}: {s:.4f}")

cv_mean = cv_scores.mean()
cv_std  = cv_scores.std()
print(f"\nCV estimate  ->  mean +/- std = {cv_mean:.4f} +/- {cv_std:.4f}")

In [ ]:
# For comparison, here is PLAIN KFold (no stratification). It ignores class balance when cutting
# folds. On this fairly balanced dataset the difference is small, but on imbalanced data plain
# KFold can produce folds with skewed class ratios and noisier scores -- which is exactly why
# StratifiedKFold is the go-to choice for classification.
kf = KFold(n_splits=k, shuffle=True, random_state=0)
kf_scores = cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=kf, scoring="accuracy")

print(f"plain KFold      : mean +/- std = {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}")
print(f"StratifiedKFold  : mean +/- std = {cv_mean:.4f} +/- {cv_std:.4f}")

## Compare: single-split spread vs CV folds

Now put both experiments on one picture:

- the **cloud of 200 single-split accuracies** from Part 1 (wide and noisy), and
- the **5 CV fold scores** with their mean line from Part 2.

Watch where the CV mean sits relative to the single-split cloud, and how tight the CV scores are.

In [ ]:
plt.figure(figsize=(10, 5))

# Row 1 (y=0): all 200 single-split accuracies as translucent dots -- the noisy cloud.
sns.stripplot(x=split_accuracies, y=["single split"] * len(split_accuracies),
              color="#6699cc", alpha=0.35, jitter=0.25, size=5)

# Row 2 (y=1): the k CV fold scores as bold dots.
sns.stripplot(x=cv_scores, y=["5-fold CV"] * len(cv_scores),
              color="#cc6666", alpha=0.9, jitter=False, size=12, marker="D")

# Reference lines: the mean of each approach.
plt.axvline(acc_mean, color="#336699", linestyle="--", label=f"single-split mean = {acc_mean:.3f}")
plt.axvline(cv_mean,  color="#993333", linestyle="-",  label=f"CV mean = {cv_mean:.3f}")

plt.title("Noisy single splits vs. stable k-fold CV")
plt.xlabel("accuracy")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Numbers side by side, for the summary below.
print(f"single split  : mean {acc_mean:.4f}, spread(min-max) {spread:.4f}")
print(f"5-fold CV     : mean {cv_mean:.4f},  std {cv_std:.4f}")

## Why CV's estimate is better — bias and variance of the *estimate*

It helps to think about the accuracy *number* itself as a measurement, which — like any measurement — has two kinds of error:

- **Variance (how jumpy the number is).** A single split reports the score from *one* random test set, so it inherits all that randomness — that's the wide cloud in the plots and the `spread` we printed. The CV mean **averages k folds**, and averaging shrinks variance. That's why the red CV dots huddle together while the blue single-split dots scatter. The CV **std** is your honest read on how much even this averaged number might still wobble.
- **Bias (is the number aimed at the right place).** With a single split you train on only 75% of the data; with k-fold you effectively train on `k-1` folds (here 80%) *and* every row eventually gets tested. Using the data more fully makes the estimate a fairer picture of how the final model — trained on everything — will behave.

### Why `std` matters, not just the mean
Two models can share the same mean accuracy but very different stds. A model reporting `0.96 ± 0.01` is far more dependable than `0.96 ± 0.06`: the second one's true performance could be noticeably better or worse depending on the data it meets. A **small std means the estimate is stable and you can trust the mean**; a large std is a warning that your score is fragile (too little data, too few folds, or a model sensitive to which rows it sees).

### Practical rules of thumb
1. **Never report accuracy from a single split as if it were the truth** — it's one draw from a noisy distribution.
2. **Use k-fold CV** (k = 5 or 10 is standard) and report **mean ± std**.
3. **For classification, use `StratifiedKFold`** so every fold keeps the class balance.
4. **Still keep a final untouched test set** for the very last, one-time check — CV is for choosing and comparing models during development; the held-out test set confirms the winner on data it has truly never seen.